# INSTALLATION NOTEBOOK

Instala os pacotes necessários para rodar todo o projeto e baixa os datasets utilizados no estudo


In [ ]:
!pip install pandas gdown #substituir por pip install -r requirements.txt no futuro

In [1]:
import pandas as pd
import os 
import pathlib
import gdown
import zipfile
import shutil
import numpy as np

WORKING_DIRECTORY = pathlib.Path(os.getcwd())
DATASETS_FILENAME = 'datasets_links.csv'

print('Running at:', WORKING_DIRECTORY)
datasets = pd.read_csv(WORKING_DIRECTORY / DATASETS_FILENAME)
datasets

Running at: /home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/src


,Name,ID
0,MoNuSeg,1ZYeUn_zl6A3X8gFSdS07WT8Jo2O2dI-z
1,PanNuke1,1U9xjLHRChWTdfqYanmta5GJpXEijVubP
2,PanNuke2,1mkGeHwPVm7a2jETQECzCOn6YW1wuEZx5
3,PanNuke3,1b9v7Y5lIJOKAwV3fnCwL8YGXsmwxdgSe
4,NuInSeg,14ZOx078bUZC_6vgvtYie2_utORZMCzig


### MoNuSeg 

In [2]:
# download and extraction
drive_id = datasets[datasets['Name']=='MoNuSeg']['ID'].item()

Raw_MoNuSeg_Path = WORKING_DIRECTORY.parent / 'data' / 'raw' / 'MoNuSeg'
File_Path = Raw_MoNuSeg_Path / 'MoNuSeg.zip'

os.makedirs(Raw_MoNuSeg_Path, exist_ok=True)
if not os.path.exists(File_Path):
    print("Baixando do Google Drive...")
    gdown.download(id=drive_id, output=str(File_Path), quiet=False)
    print("Download concluído!")
else:
    print("Dataset .zip já encontrado. Pulando o download.")

with zipfile.ZipFile(File_Path, 'r') as zip_ref:
        zip_ref.extractall(str(Raw_MoNuSeg_Path))

print('Dataset extraido para', Raw_MoNuSeg_Path)

Dataset .zip já encontrado. Pulando o download.


Dataset extraido para /home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/raw/MoNuSeg


In [ ]:
#  SENDING IMAGES TO INTERIM FOLDER 
extracted_dir = WORKING_DIRECTORY.parent / 'data' / 'raw' / 'MoNuSeg' / 'MoNuSeg 2018 Training Data'
Interim_dir = WORKING_DIRECTORY.parent / 'data' / 'interim' / 'MoNuSeg'
os.makedirs(Interim_dir, exist_ok=True)
Interim_images_dir = Interim_dir / 'Images'
os.makedirs(Interim_images_dir, exist_ok=True)
Interim_masks_dir = Interim_dir / 'Masks'
os.makedirs(Interim_masks_dir, exist_ok=True)

dir_content = os.listdir(extracted_dir / 'Tissue Images')
dir_content = [i.replace('.tif','') for i in dir_content]
dir_content_df = pd.DataFrame({'FileName':dir_content})
dir_content_df = dir_content_df.sort_values(by='FileName') #Sorting to keep integrity
dir_content_df.to_csv(Interim_dir / 'labels.csv',index=False)
for file in dir_content:
    img_name = file + '.tif'
    mask_name = file + '.xml'
    shutil.copy(extracted_dir / 'Tissue Images' / img_name, Interim_images_dir / img_name)
    shutil.copy(extracted_dir / 'Annotations' / mask_name, Interim_masks_dir / mask_name)

### PanNuke


In [ ]:
#download and extraction
drive_ids = [datasets[datasets['Name']==f'PanNuke{i}']['ID'].item() for i in range(1,4)]

Raw_PanNuke_Path = WORKING_DIRECTORY.parent / 'data' / 'raw' / 'PanNuke'
File_Paths = [Raw_PanNuke_Path / f'PanNuke{i}.zip' for i in range(1,4)]

os.makedirs(Raw_PanNuke_Path, exist_ok=True)

for i in range(3):
    if not os.path.exists(File_Paths[i]):
        print("Baixando do Google Drive...")
        gdown.download(id=drive_ids[i], output=str(File_Paths[i]), quiet=False)
        print("Download concluído!")
    else:
        print("Dataset .zip já encontrado. Pulando o download.")

    with zipfile.ZipFile(File_Paths[i], 'r') as zip_ref:
            zip_ref.extractall(str(Raw_PanNuke_Path))

    print('Dataset extraido para', Raw_PanNuke_Path)

Dataset .zip já encontrado. Pulando o download.
Dataset extraido para /home/vinicius/IA901/Projeto-IA901/Datasets/Raw/PanNuke
Dataset .zip já encontrado. Pulando o download.
Dataset extraido para /home/vinicius/IA901/Projeto-IA901/Datasets/Raw/PanNuke
Dataset .zip já encontrado. Pulando o download.
Dataset extraido para /home/vinicius/IA901/Projeto-IA901/Datasets/Raw/PanNuke


In [ ]:
#  SENDING IMAGES TO INTERIM FOLDER 
extracted_dirs = [WORKING_DIRECTORY.parent / 'data' / 'raw' / 'PanNuke' / f'Fold {i}' for i in range(1,4)] 
Interim_dir = WORKING_DIRECTORY.parent / 'data' / 'interim' / 'PanNuke'
os.makedirs(Interim_dir, exist_ok=True)
Interim_images_dir = Interim_dir / 'Images'
os.makedirs(Interim_images_dir, exist_ok=True)
Interim_masks_dir = Interim_dir / 'Masks'
os.makedirs(Interim_masks_dir, exist_ok=True)

lista_images = []
lista_types = []
lista_masks = []

for idx, dir in enumerate(extracted_dirs):
    lista_images.append(np.load(dir / 'images' / f'fold{idx+1}' / 'images.npy'))
    lista_types.append(np.load(dir / 'images' / f'fold{idx+1}' / 'types.npy'))
    lista_masks.append(np.load(dir / 'masks'  / f'fold{idx+1}' / 'masks.npy'))

images_finais = np.concatenate(lista_images, axis=0)
types_finais = np.concatenate(lista_types, axis=0)
masks_finais = np.concatenate(lista_masks, axis=0)

types_df = pd.DataFrame({'FileName':[i for i in range(types_finais.shape[0])], 'Type':types_finais})
types_df.to_csv(Interim_dir / 'labels.csv',index=False)

np.save(Interim_images_dir / 'images.npy',images_finais)
np.save(Interim_masks_dir / 'masks.npy', masks_finais)


### NuInSeg

In [ ]:
#download and extraction

drive_id = datasets[datasets['Name']=='NuInSeg']['ID'].item()

Raw_NuInSeg_Path = WORKING_DIRECTORY.parent / 'data' / 'raw' / 'NuInSeg'
File_Path = Raw_NuInSeg_Path / 'NuInSeg.zip'

os.makedirs(Raw_NuInSeg_Path, exist_ok=True)
if not os.path.exists(File_Path):
    print("Baixando do Google Drive...")
    gdown.download(id=drive_id, output=str(File_Path), quiet=False)
    print("Download concluído!")
else:
    print("Dataset .zip já encontrado. Pulando o download.")

with zipfile.ZipFile(File_Path, 'r') as zip_ref:
        zip_ref.extractall(str(Raw_NuInSeg_Path))

print('Dataset extraido para', Raw_NuInSeg_Path)

Dataset .zip já encontrado. Pulando o download.
Dataset extraido para /home/vinicius/IA901/Projeto-IA901/Datasets/Raw/NuInSeg


In [ ]:
#  SENDING IMAGES TO INTERIM FOLDER 
extracted_dir = WORKING_DIRECTORY.parent / 'data' / 'raw' / 'NuInSeg' / 'NuInSeg'
Interim_dir = WORKING_DIRECTORY.parent / 'data' / 'interim' / 'NuInSeg'
os.makedirs(Interim_dir, exist_ok=True)
Interim_images_dir = Interim_dir / 'Images'
os.makedirs(Interim_images_dir, exist_ok=True)
Interim_masks_dir = Interim_dir / 'Masks'
os.makedirs(Interim_masks_dir, exist_ok=True)

tissues = []
images = []

dir_content = os.listdir(extracted_dir)
for dir in dir_content:
    current_dir = extracted_dir / dir  
    in_folder = os.listdir(current_dir / 'tissue images')
    for img in in_folder:
        tissues.append(dir)
        images.append(img.replace('.png',''))
        shutil.copy(current_dir / 'tissue images' / img, Interim_images_dir / img)
        shutil.copy(current_dir / 'mask binary' /img, Interim_masks_dir / img) 
    
dir_content_df = pd.DataFrame({'FileName':images, 'Tissue':tissues})
dir_content_df = dir_content_df.sort_values(by='FileName') #Sorting to keep integrity
dir_content_df.to_csv(Interim_dir / 'labels.csv',index=False)